In [2]:
!pip install catboost

ERROR: Target path exists but is not a directory, will not continue.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor

## **CatBoost Regression**

In [ ]:
# Load dataset
df = pd.read_excel('TCI_sas (1).xlsx', sheet_name="Sheet1", usecols=["Hrd", "milkperiod", "zdate", "zdate_month", "firstmilk",
                                                                       "firstmilkdays", "prelendays", "drylendays", "milkdays",
                                                                       "grp", "curent_Milk305", "previous_Milk305",
                                                                       "firstmilk_previous", "SCS_305"])

In [ ]:
# Filter out rows with missing values instead of filling with mean
df = df.dropna(subset=['drylendays', 'milkdays'])

# Filter invalid data
df = df[df['milkdays'] >= 0]
df = df[df['drylendays'] >= 0]
df = df[df['previous_Milk305'] >= 0]
df = df[df['firstmilk_previous'] >= 0]

# Feature Engineering
df['is_abortion'] = (df['prelendays'] < 260).astype(int)
df = pd.get_dummies(df, columns=['grp'], prefix='grp')

In [ ]:
df.head()

In [ ]:
# Prepare features and target
numerical_columns = [col for col in df.columns if col not in ['firstmilk', 'curent_Milk305']]
X = df[numerical_columns]
y = df["firstmilk"].astype(float)

# Split into train, validation, and test sets
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)

# Scale the data
scaler = RobustScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
def calculate_mpe(y_true, y_pred):
    # Avoid division by zero
    mask = y_true != 0
    return np.mean((y_true[mask] - y_pred[mask]) / y_true[mask] * 100)

def calculate_smape(y_true, y_pred):
    # Avoid division by zero
    numerator = np.abs(y_true - y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominator != 0
    return np.mean(numerator[mask] / denominator[mask] * 100)

def calculate_sdr(y_true, y_pred):
    return np.std(y_pred) / np.std(y_true)

In [ ]:
# Define and train CatBoost Model
print("\n=== CatBoost Model (with missing data dropped) ===")
model = CatBoostRegressor(iterations=1000, learning_rate=0.1, depth=6, verbose=100)
model.fit(X_train, y_train, eval_set=(X_val, y_val))

In [ ]:
y_test_pred = model.predict(X_test)

# Calculate metrics
test_r2 = r2_score(y_test, y_test_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_mpe = calculate_mpe(y_test, y_test_pred)
test_smape = calculate_smape(y_test, y_test_pred)
test_sdr = calculate_sdr(y_test, y_test_pred)

# Print results
print("Test Set (CatBoost Model):")
print(f"R²    : {test_r2:.4f}")
print(f"MAE   : {test_mae:.4f}")
print(f"RMSE  : {test_rmse:.4f}")
print(f"MPE   : {test_mpe:.4f}")
print(f"sMAPE : {test_smape:.4f}")
print(f"SDR   : {test_sdr:.4f}")

In [ ]:
# Predict on all sets
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)

# Calculate metrics
train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_mae = mean_absolute_error(y_train, y_train_pred)
val_mae = mean_absolute_error(y_val, y_val_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
val_mse = mean_squared_error(y_val, y_val_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

# Print results
print("Train Set (CatBoost Model):")
print(f"R²   : {train_r2:.4f}")
print(f"MAE  : {train_mae:.4f}")
print(f"MSE  : {train_mse:.4f}")
print("\nValidation Set (CatBoost Model):")
print(f"R²   : {val_r2:.4f}")
print(f"MAE  : {val_mae:.4f}")
print(f"MSE  : {val_mse:.4f}")
print("\nTest Set (CatBoost Model):")
print(f"R²   : {test_r2:.4f}")
print(f"MAE  : {test_mae:.4f}")
print(f"MSE  : {test_mse:.4f}")

In [ ]:
# Overfitting Check
r2_gap = train_r2 - val_r2
print("\nOverfitting Analysis (CatBoost Model):")
print("=" * 50)
if r2_gap > 0.05:
    print(f"Warning: Potential Overfitting Detected! Train-Val R² Gap ({r2_gap:.4f}) is larger than threshold (0.05).")
else:
    print(f"No significant overfitting based on Train-Val R² Gap ({r2_gap:.4f}).")

In [ ]:
# Save results to existing CSV file

comparison_df = pd.read_csv('model_comparison_table (1).csv', index_col=0)
new_results = {
    'CatBoost_Initial': {
        'Train_R2': train_r2, 'Train_MAE': train_mae, 'Train_MSE': train_mse,
        'Val_R2': val_r2, 'Val_MAE': val_mae, 'Val_MSE': val_mse,
        'Test_R2': test_r2, 'Test_MAE': test_mae, 'Test_MSE': test_mse
    }
}
new_df = pd.DataFrame.from_dict(new_results, orient='index')
comparison_df = pd.concat([comparison_df, new_df])
comparison_df.to_csv('model_comparison_table (1).csv', index=True)
print("\n=== Updated Model Comparison Table ===")
print(comparison_df)

## **Optimized CatBoost Regression**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor
import numpy as np

In [ ]:
# Load dataset
df = pd.read_excel('TCI_sas (1).xlsx', sheet_name="Sheet1", usecols=["Hrd", "milkperiod", "zdate", "zdate_month", "firstmilk",
                                                                       "firstmilkdays", "prelendays", "drylendays", "milkdays",
                                                                       "grp", "curent_Milk305", "previous_Milk305",
                                                                       "firstmilk_previous", "SCS_305"])

In [ ]:
# Filter out rows with missing values instead of filling with mean
df = df.dropna(subset=['drylendays', 'milkdays'])

# Filter invalid data
df = df[df['milkdays'] >= 0]
df = df[df['drylendays'] >= 0]
df = df[df['previous_Milk305'] >= 0]
df = df[df['firstmilk_previous'] >= 0]

# Feature Engineering
df['is_abortion'] = (df['prelendays'] < 260).astype(int)
df = pd.get_dummies(df, columns=['grp'], prefix='grp')

# Prepare features and target
numerical_columns = [col for col in df.columns if col not in ['firstmilk', 'curent_Milk305']]
X = df[numerical_columns]
y = df["firstmilk"].astype(float)

# Split into train, validation, and test sets
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)

# Scale the data
scaler = RobustScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
# Define parameter grid for GridSearchCV
param_grid = {
    'iterations': [500, 1000, 1500],
    'learning_rate': [0.01, 0.1, 0.2],
    'depth': [4, 6, 8],
    'l2_leaf_reg': [1, 3, 5]
}

# Initialize CatBoost model with GPU support
model = CatBoostRegressor(
    verbose=100,
    random_state=42,
    early_stopping_rounds=50,
    task_type="GPU",  # Enable GPU support
    devices='0'       # Specify GPU device (0 for the first GPU)
)

# Perform Grid Search
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='r2', n_jobs=1, verbose=100)
# Note: n_jobs=-1 is not supported with GPU in CatBoost, so set to 1
grid_search.fit(X_train, y_train, eval_set=(X_val, y_val))

# Get best model
best_model = grid_search.best_estimator_
print("\n=== Best Parameters ===")
print(grid_search.best_params_)

In [ ]:
# Predict on test set with best model
y_test_pred_optimized = best_model.predict(X_test_optimized)

# Calculate metrics
test_r2_optimized = r2_score(y_test_optimized, y_test_pred_optimized)
test_mae_optimized = mean_absolute_error(y_test_optimized, y_test_pred_optimized)
test_mse_optimized = mean_squared_error(y_test_optimized, y_test_pred_optimized)
test_rmse_optimized = np.sqrt(test_mse_optimized)
test_mpe_optimized = calculate_mpe(y_test_optimized, y_test_pred_optimized)
test_smape_optimized = calculate_smape(y_test_optimized, y_test_pred_optimized)
test_sdr_optimized = calculate_sdr(y_test_optimized, y_test_pred_optimized)

# Print results
print("Test Set (Optimized CatBoost Model):")
print(f"R²    : {test_r2_optimized:.4f}")
print(f"MAE   : {test_mae_optimized:.4f}")
print(f"RMSE  : {test_rmse_optimized:.4f}")
print(f"MPE   : {test_mpe_optimized:.4f}")
print(f"sMAPE : {test_smape_optimized:.4f}")
print(f"SDR   : {test_sdr_optimized:.4f}")

In [ ]:
# Predict on all sets with best model
y_train_pred = best_model.predict(X_train)
y_val_pred = best_model.predict(X_val)
y_test_pred = best_model.predict(X_test)

# Calculate metrics
train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_mae = mean_absolute_error(y_train, y_train_pred)
val_mae = mean_absolute_error(y_val, y_val_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
val_mse = mean_squared_error(y_val, y_val_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

# Print results
print("\nTrain Set (Optimized CatBoost Model):")
print(f"R²   : {train_r2:.4f}")
print(f"MAE  : {train_mae:.4f}")
print(f"MSE  : {train_mse:.4f}")
print("\nValidation Set (Optimized CatBoost Model):")
print(f"R²   : {val_r2:.4f}")
print(f"MAE  : {val_mae:.4f}")
print(f"MSE  : {val_mse:.4f}")
print("\nTest Set (Optimized CatBoost Model):")
print(f"R²   : {test_r2:.4f}")
print(f"MAE  : {test_mae:.4f}")
print(f"MSE  : {test_mse:.4f}")

In [ ]:
# Overfitting Check
r2_gap = train_r2 - val_r2
print("\nOverfitting Analysis (Optimized CatBoost Model):")
print("=" * 50)
if r2_gap > 0.05:
    print(f"Warning: Potential Overfitting Detected! Train-Val R² Gap ({r2_gap:.4f}) is larger than threshold (0.05).")
else:
    print(f"No significant overfitting based on Train-Val R² Gap ({r2_gap:.4f}).")

In [ ]:
# Save results to existing CSV file

comparison_df = pd.read_csv('model_comparison_table (1) (1).csv', index_col=0)
new_results = {
    'CatBoost_Optimized': {
        'Train_R2': train_r2, 'Train_MAE': train_mae, 'Train_MSE': train_mse,
        'Val_R2': val_r2, 'Val_MAE': val_mae, 'Val_MSE': val_mse,
        'Test_R2': test_r2, 'Test_MAE': test_mae, 'Test_MSE': test_mse
    }
}
new_df = pd.DataFrame.from_dict(new_results, orient='index')
comparison_df = pd.concat([comparison_df, new_df])
comparison_df.to_csv('model_comparison_table (1) (1).csv', index=True)
print("\n=== Updated Model Comparison Table ===")
print(comparison_df)

## **CatBoost Regression with same old standard scale and fill miss value with median**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor
import numpy as np

In [ ]:
# Load dataset
df = pd.read_excel('TCI_sas (1).xlsx', sheet_name="Sheet1", usecols=["milkperiod", "zdate", "zdate_month", "firstmilk",
                                                                       "firstmilkdays", "prelendays", "drylendays", "milkdays",
                                                                       "previous_Milk305",
                                                                       "firstmilk_previous", "SCS_305"])

In [ ]:
df['drylendays'] = df['drylendays'].fillna(df['drylendays'].mean())
df['milkdays'] = df['milkdays'].fillna(df['milkdays'].mean())

In [ ]:
# Separate features and target
X = df.drop(columns=["firstmilk"])
y = df["firstmilk"].astype(float)

# First split: separate test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# Second split: separate train (70%) and validation (15%) from remaining data
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)  # 0.1765 ~ 15/(100-15)


In [ ]:
numerical_columns = ["milkperiod","zdate","zdate_month", "firstmilkdays", "prelendays", "drylendays", "milkdays",
                     "previous_Milk305", "firstmilk_previous", "SCS_305"]

scaler = StandardScaler()

# Fit scaler on training data only
scaler.fit(X_train[numerical_columns])

# Transform train, validation, and test sets
X_train[numerical_columns] = scaler.transform(X_train[numerical_columns])
X_val[numerical_columns] = scaler.transform(X_val[numerical_columns])
X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])

In [ ]:
# Define and train CatBoost Model
print("\n=== CatBoost Model (same old model scale and missing null value) ===")
model = CatBoostRegressor(iterations=1000, learning_rate=0.1, depth=6, verbose=100)
model.fit(X_train, y_train, eval_set=(X_val, y_val))

In [ ]:
# Define and train CatBoost Model
print("\n=== CatBoost Model (same old model scale and missing null value) ===")
model_standard = CatBoostRegressor(iterations=500, learning_rate=0.2, depth=8, verbose=100)
model_standard.fit(X_train_standard, y_train_standard, eval_set=(X_val_standard, y_val_standard))

# Predict on test set
y_test_pred_standard = model_standard.predict(X_test_standard)

# Calculate metrics
test_r2_standard = r2_score(y_test_standard, y_test_pred_standard)
test_mae_standard = mean_absolute_error(y_test_standard, y_test_pred_standard)
test_mse_standard = mean_squared_error(y_test_standard, y_test_pred_standard)
test_rmse_standard = np.sqrt(test_mse_standard)
test_mpe_standard = calculate_mpe(y_test_standard, y_test_pred_standard)
test_smape_standard = calculate_smape(y_test_standard, y_test_pred_standard)
test_sdr_standard = calculate_sdr(y_test_standard, y_test_pred_standard)

# Print results
print("Test Set (CatBoost Model):")
print(f"R²    : {test_r2_standard:.4f}")
print(f"MAE   : {test_mae_standard:.4f}")
print(f"RMSE  : {test_rmse_standard:.4f}")
print(f"MPE   : {test_mpe_standard:.4f}")
print(f"sMAPE : {test_smape_standard:.4f}")
print(f"SDR   : {test_sdr_standard:.4f}")

In [ ]:
# Predict on all sets
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)

# Calculate metrics
train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_mae = mean_absolute_error(y_train, y_train_pred)
val_mae = mean_absolute_error(y_val, y_val_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
val_mse = mean_squared_error(y_val, y_val_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

# Print results
print("Train Set (CatBoost Model):")
print(f"R²   : {train_r2:.4f}")
print(f"MAE  : {train_mae:.4f}")
print(f"MSE  : {train_mse:.4f}")
print("\nValidation Set (CatBoost Model):")
print(f"R²   : {val_r2:.4f}")
print(f"MAE  : {val_mae:.4f}")
print(f"MSE  : {val_mse:.4f}")
print("\nTest Set (CatBoost Model):")
print(f"R²   : {test_r2:.4f}")
print(f"MAE  : {test_mae:.4f}")
print(f"MSE  : {test_mse:.4f}")

In [ ]:
# Overfitting Check
r2_gap = train_r2 - val_r2
print("\nOverfitting Analysis (CatBoost Model):")
print("=" * 50)
if r2_gap > 0.05:
    print(f"Warning: Potential Overfitting Detected! Train-Val R² Gap ({r2_gap:.4f}) is larger than threshold (0.05).")
else:
    print(f"No significant overfitting based on Train-Val R² Gap ({r2_gap:.4f}).")

In [ ]:
# Save results to existing CSV file

comparison_df = pd.read_csv('model_comparison_table (1).csv', index_col=0)
new_results = {
    'CatBoost_Initial (same old model scale and missing null value)': {
        'Train_R2': train_r2, 'Train_MAE': train_mae, 'Train_MSE': train_mse,
        'Val_R2': val_r2, 'Val_MAE': val_mae, 'Val_MSE': val_mse,
        'Test_R2': test_r2, 'Test_MAE': test_mae, 'Test_MSE': test_mse
    }
}
new_df = pd.DataFrame.from_dict(new_results, orient='index')
comparison_df = pd.concat([comparison_df, new_df])
comparison_df.to_csv('model_comparison_table (1).csv', index=True)
print("\n=== Updated Model Comparison Table ===")
print(comparison_df)

In [ ]:
## **Print results into a new CSV file**
# Function to print existing CSV content
def print_csv_content(file_path):
    try:
        existing_df = pd.read_csv(file_path, index_col=0)
        return existing_df
    except FileNotFoundError:
        print("\nNo existing test evaluation table found.")
        return pd.DataFrame()

# Load existing CSV content from new file (if exists)
new_file_path = 'test_evaluation_metrics.csv'
existing_df = print_csv_content(new_file_path)

# Save results to a new CSV file
catboost_results = {
    'CatBoost_Initial': {
        'Test_R2': test_r2, 'Test_MAE': test_mae, 'Test_RMSE': test_rmse,
        'Test_MPE': test_mpe, 'Test_sMAPE': test_smape, 'Test_SDR': test_sdr
    }
}
catboost_df = pd.DataFrame.from_dict(catboost_results, orient='index')

catboost_optimized_results = {
    'CatBoost_Optimized': {
        'Test_R2': test_r2_optimized, 'Test_MAE': test_mae_optimized, 'Test_RMSE': test_rmse_optimized,
        'Test_MPE': test_mpe_optimized, 'Test_sMAPE': test_smape_optimized, 'Test_SDR': test_sdr_optimized
    }
}
catboost_optimized_df = pd.DataFrame.from_dict(catboost_optimized_results, orient='index')

catboost_standard_results = {
    'CatBoost_Initial_Standard': {
        'Test_R2': test_r2_standard, 'Test_MAE': test_mae_standard, 'Test_RMSE': test_rmse_standard,
        'Test_MPE': test_mpe_standard, 'Test_sMAPE': test_smape_standard, 'Test_SDR': test_sdr_standard
    }
}
catboost_standard_df = pd.DataFrame.from_dict(catboost_standard_results, orient='index')

# Combine new results
new_results_df = pd.concat([catboost_df, catboost_optimized_df, catboost_standard_df])

# If existing data exists, append it; otherwise, start with new results
if not existing_df.empty:
    comparison_df = pd.concat([new_results_df, existing_df])
else:
    comparison_df = new_results_df

# Save to the new file
comparison_df.to_csv(new_file_path, index=True)
print("\n=== Updated Test Evaluation Metrics Saved to test_evaluation_metrics.csv ===")
print(comparison_df)